In [1]:
# Wheat Crop Yield Prediction
import pandas as pd
import numpy as np

column_names = ['State', 'District', 'Year', 'Area (Hectare)', 'Production (Tonnes)', 'Yield (Tonne/Hectare)']

# Load CSV slicing only relevant first 6 columns, skipping initial 3 rows
data = pd.read_csv('wheat_yield_data.csv', skiprows=3, usecols=range(6), names=column_names)

# Remove duplicate header rows if any
data = data[~data['State'].astype(str).str.contains('State', na=False)]

# Clean Yields by converting to numeric (removing commas if any)
data['Yield (Tonne/Hectare)'] = pd.to_numeric(data['Yield (Tonne/Hectare)'].astype(str).str.replace(',', ''), errors='coerce')
data.dropna(subset=['Yield (Tonne/Hectare)'], inplace=True)

# Extract Year integer from range string
def extract_year(year):
    if isinstance(year, str) and '-' in year:
        return int(year.split('-')[0].strip())
    try:
        return int(year)
    except:
        return np.nan

data['Year'] = data['Year'].apply(extract_year)
data.dropna(subset=['Year'], inplace=True)

print(data.head())
print(data.shape)


       State     District  Year Area (Hectare) Production (Tonnes)  \
1  1. Punjab  1. Amritsar  1997      349000.00          1318000.00   
2  1. Punjab  1. Amritsar  1998      214000.00           862000.00   
3  1. Punjab  1. Amritsar  1999      358000.00          1738000.00   
4  1. Punjab  1. Amritsar  2000      361000.00          1690000.00   
5  1. Punjab  1. Amritsar  2001      363000.00          1700000.00   

   Yield (Tonne/Hectare)  
1                   3.78  
2                   4.03  
3                   4.85  
4                   4.68  
5                   4.68  
(516, 6)


In [78]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import numpy as np

# Define features and target
X = data.drop(columns=['Yield (Tonne/Hectare)'])
y = data['Yield (Tonne/Hectare)']

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()

# Preprocessing steps for numeric and categorical features
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

# Build the pipeline with a Random Forest regressor
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
pipeline.fit(X_train, y_train)

# Predict and evaluate
y_pred = pipeline.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.3f}")
print(f"R^2: {r2:.3f}")

# Save the trained pipeline for future use
joblib.dump(pipeline, 'crop_yield_model.pkl')


RMSE: 0.283
R^2: 0.659


['crop_yield_model.pkl']

In [80]:
pip install xgboost

   ---------------------------------------- 0.0/56.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/56.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/56.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/56.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/56.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/56.8 MB ? eta -:--:--
    --------------------------------------- 0.8/56.8 MB 1.2 MB/s eta 0:00:49
    --------------------------------------- 0.8/56.8 MB 1.2 MB/s eta 0:00:49
    --------------------------------------- 1.0/56.8 MB 968.5 kB/s eta 0:00:58
    --------------------------------------- 1.3/56.8 MB 1.0 MB/s eta 0:00:55
   - -------------------------------------- 1.8/56.8 MB 1.2 MB/s eta 0:00:45
   - -------------------------------------- 2.1/56.8 MB 1.4 MB/s eta 0:00:41
   -- ------------------------------------- 2.9/56.8 MB 1.6 MB/s eta 0:00:35
   -- ---------------------------------


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: C:\Users\tanma\.conda\envs\myenv\python.exe -m pip install --upgrade pip


In [81]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# Load and clean data (same as before)
column_names = ['State', 'District', 'Year', 'Area (Hectare)', 'Production (Tonnes)', 'Yield (Tonne/Hectare)']
data = pd.read_csv('wheat_yield_data.csv', skiprows=3, usecols=range(6), names=column_names)
data = data[~data['State'].astype(str).str.contains('State', na=False)]
data['Yield (Tonne/Hectare)'] = pd.to_numeric(data['Yield (Tonne/Hectare)'].astype(str).str.replace(',', ''), errors='coerce')
data.dropna(subset=['Yield (Tonne/Hectare)'], inplace=True)

def extract_year(year):
    if isinstance(year, str) and '-' in year:
        return int(year.split('-')[0].strip())
    try:
        return int(year)
    except:
        return np.nan
data['Year'] = data['Year'].apply(extract_year)
data.dropna(subset=['Year'], inplace=True)

# Outlier detection and handling (e.g., winsorizing Yield)
q_low = data['Yield (Tonne/Hectare)'].quantile(0.01)
q_hi  = data['Yield (Tonne/Hectare)'].quantile(0.99)
data['Yield (Tonne/Hectare)'] = np.where(data['Yield (Tonne/Hectare)'] < q_low, q_low, data['Yield (Tonne/Hectare)'])
data['Yield (Tonne/Hectare)'] = np.where(data['Yield (Tonne/Hectare)'] > q_hi, q_hi, data['Yield (Tonne/Hectare)'])

X = data.drop(columns=['Yield (Tonne/Hectare)'])
y = data['Yield (Tonne/Hectare)']

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

xgb_model = XGBRegressor(random_state=42, objective='reg:squarederror')

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb_model)
])

# Hyperparameter tuning with GridSearchCV
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.01, 0.1],
    'model__subsample': [0.7, 1.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best RMSE (CV):", -grid_search.best_score_)

y_pred = grid_search.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Test RMSE: {rmse:.3f}")
print(f"Test R^2: {r2:.3f}")

# Save best model pipeline
joblib.dump(grid_search.best_estimator_, 'crop_yield_xgb_model.pkl')


Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': 5, 'model__n_estimators': 200, 'model__subsample': 0.7}
Best RMSE (CV): 0.23479998880271577
Test RMSE: 0.239
Test R^2: 0.755


['crop_yield_xgb_model.pkl']